## Question 4 — Causal self-attention

Autoregressive generation predicts one token at a time, so position $i$ may use only itself and earlier tokens, not future positions $j>i$. Training still feeds the full target in one pass, so a causal mask $M$ is added to the scores $QK^\top/\sqrt{d}$ before softmax. $M_{ij}=0$ for $j\le i$ leaves self and past scores unchanged; $M_{ij}=-\infty$ for $j>i$ makes those softmax weights $0$. A $4\times4$ mask (rows are queries, columns are keys) is


$$
M=
\begin{bmatrix}
\phantom{-}0 & -\infty & -\infty & -\infty \\
\phantom{-}0 & \phantom{-}0 & -\infty & -\infty \\
\phantom{-}0 & \phantom{-}0 & \phantom{-}0 & -\infty \\
\phantom{-}0 & \phantom{-}0 & \phantom{-}0 & \phantom{-}0
\end{bmatrix}.
$$

Position 1 attends only to itself; position 4 attends to all four. The diagonal stays $0$ because a token may attend to itself.

**Imports.** NumPy builds the scores, the causal mask, and the check.

In [5]:
import numpy as np

**Stable softmax.** Subtract the row max before the exponential. A masked score of $-\infty$ then becomes weight $0$.

In [6]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)

**Causal self-attention.** Scores are $QK^\top/\sqrt{d_k}$. Entries strictly above the diagonal are set to $-\infty$ before softmax, so query $i$ attends only to keys $j\le i$.

In [7]:
def causal_self_attention(Q, K, V):
    """Causal self-attention. Q, K are (n, d_k); V is (n, d_v)."""
    Q, K, V = (np.asarray(z, dtype=float) for z in (Q, K, V))
    if (
        Q.ndim != 2
        or K.ndim != 2
        or V.ndim != 2
        or Q.shape[0] != K.shape[0]
        or K.shape[0] != V.shape[0]
        or Q.shape[1] != K.shape[1]
    ):
        raise ValueError("Causal self-attention requires matching sequence lengths")
    n = Q.shape[0]
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    future = np.triu(np.ones((n, n), dtype=bool), k=1)
    masked_scores = np.where(future, -np.inf, scores)
    weights = stable_softmax(masked_scores, axis=-1)
    return weights @ V, weights

**Example tokens.** Three positions. Queries, keys, and values are this matrix, so $d_k = d_v = 2$.

In [8]:
X4 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

**$3\times 3$ mask.** $0$ on and below the diagonal, $-\infty$ strictly above it. `k=1` leaves the diagonal unmasked.

In [10]:
print("Q4 mask:\n", np.where(np.triu(np.ones((3, 3), dtype=bool), k=1), -np.inf, 0.0))

Q4 mask:
 [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]


**Weights and output.** Each row of the weights is a distribution over positions this query is allowed to see.

In [11]:
out4, weights4 = causal_self_attention(X4, X4, X4)
print("Q4 weights:\n", weights4)
print("Q4 output:\n", out4)

Q4 weights:
 [[1.         0.         0.        ]
 [0.33023845 0.66976155 0.        ]
 [0.24825508 0.24825508 0.50348984]]
Q4 output:
 [[1.         0.        ]
 [0.33023845 0.66976155]
 [0.75174492 0.75174492]]


**Check.** Future weights are $0$, and each query row still sums to $1$.

In [12]:
assert np.all(weights4[np.triu_indices(3, k=1)] == 0)
np.testing.assert_allclose(weights4.sum(axis=1), 1)
print("Causal mask check passed.")

Causal mask check passed.


(d) Causal masking encodes time order and blocks future target positions. Padding masking blocks nonexistent/padded input positions, regardless of whether those positions are earlier or later.